# ARC-AGI-3 — Step 3: State-Graph Frontier Search (BFS)

Builds on Step 2 (action-effect change-rate tracking). Adds the other half of the official preview competition's winning pattern: 2nd place (Blind Squirrel) built **a directed state graph from observed frames** and searched it.

**Important scope note on what's actually achievable here:** some public solver write-ups do full MCTS/A* with simulated rollouts, by loading the game's own Python source file and instantiating private copies of it to try many futures before committing to one. That only works for games whose source file is discoverable on disk -- true for the public/offline practice environments, but *not verified* to be true for the private evaluation games used for real scoring, since we have no confirmation their engine classes are introspectable the same way. Rather than build on that unverified assumption, this step only uses what's 100% guaranteed by the documented live API: real actions, real resulting frames. No simulated/offline rollouts.

What it does instead -- real BFS frontier-expansion, built from live play:
1. Treat each distinct frame (by hash) as a graph node, and each action taken as    a directed edge to the resulting node -- this mirrors the ARC Prize    Foundation's own state-graph methodology (their technical report, section    3.5.2: "each node corresponds to a unique environment state ... each edge    corresponds to a valid player action").
2. From the current node, always prefer an action we haven't tried *from this    exact state* yet -- classic BFS frontier expansion, explore all edges from a    node before revisiting it.
3. If every action from this state has already been tried, fall back to Step 2's    change-rate preference, with higher randomness to break out of cycles.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
import hashlib
import random
import time
from typing import Any, Dict, List, Optional, Set, Tuple

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

REGION_SIZE = 8  # bucket the 64x64 coordinate grid into 8x8 regions


class MyAgent(Agent):
    """Step 3: real-play state-graph BFS on top of Step 2's action-effect tracking.

    Graph: node = frame hash, edge = (action taken from that node) -> resulting
    node. Built entirely from actions actually taken (no simulated rollouts --
    see the notebook markdown cell for why). Each turn, prefer an action not
    yet tried from the *current* node (BFS frontier expansion); if the current
    node is fully explored, fall back to Step 2's global change-rate heuristic.
    """

    MAX_ACTIONS = 80
    EPSILON = 0.15          # normal random-exploration rate
    STUCK_EPSILON = 0.5     # higher random rate once a node is fully explored

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)
        # Step 2: action_id -> [times_tried, times_it_changed_the_frame]
        self.action_stats: Dict[int, List[int]] = {}
        # Step 2: (region_x, region_y) -> [times_tried, times_it_changed_the_frame]
        self.region_stats: Dict[Tuple[int, int], List[int]] = {}
        # Step 3: directed state graph. node_hash -> {action_id: next_node_hash}
        self.graph: Dict[str, Dict[int, str]] = {}
        # Step 3: node_hash -> set of action_ids already tried from that node
        self.tried_from: Dict[str, Set[int]] = {}
        self.last_hash: Optional[str] = None
        self.pending_id: Optional[int] = None
        self.pending_region: Optional[Tuple[int, int]] = None

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _frame_hash(self, latest_frame: FrameData) -> str:
        return hashlib.md5(str(latest_frame.frame).encode()).hexdigest()

    def _record_outcome(self, current_hash: str) -> None:
        """Attribute the frame-change outcome AND the graph edge to whatever
        we did last turn, now that we can see where it led."""
        if self.pending_id is None or self.last_hash is None:
            return
        changed = current_hash != self.last_hash

        # Step 2 bookkeeping (action/region change-rate)
        stats = self.action_stats.setdefault(self.pending_id, [0, 0])
        stats[0] += 1
        stats[1] += int(changed)
        if self.pending_region is not None:
            rstats = self.region_stats.setdefault(self.pending_region, [0, 0])
            rstats[0] += 1
            rstats[1] += int(changed)

        # Step 3 bookkeeping (graph edge + per-node tried set)
        self.graph.setdefault(self.last_hash, {})[self.pending_id] = current_hash
        self.tried_from.setdefault(self.last_hash, set()).add(self.pending_id)

    def _score(self, stats_dict: dict, key: Any) -> float:
        tried, changed = stats_dict.get(key, [0, 0])
        if tried == 0:
            return 1.0
        return changed / tried

    def _pick_by_score(self, stats_dict: dict, keys: list, epsilon: float) -> Any:
        if random.random() < epsilon:
            return random.choice(keys)
        best_score = max(self._score(stats_dict, k) for k in keys)
        best_keys = [k for k in keys if self._score(stats_dict, k) == best_score]
        return random.choice(best_keys)

    def _pick_action(self, node_hash: str, legal_ids: list) -> int:
        """BFS frontier rule: prefer an action not yet tried from this exact
        node. Only fall back to the Step 2 change-rate heuristic once every
        legal action from this node has already been explored."""
        tried_here = self.tried_from.get(node_hash, set())
        untried = [a for a in legal_ids if a not in tried_here]
        if untried:
            return random.choice(untried)
        # Fully explored from here: fall back to global change-rate, more randomness
        return self._pick_by_score(self.action_stats, legal_ids, self.STUCK_EPSILON)

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.last_hash = None
            self.pending_id = None
            self.pending_region = None
            return GameAction.RESET

        current_hash = self._frame_hash(latest_frame)
        self._record_outcome(current_hash)

        legal_ids = [a for a in latest_frame.available_actions if a != GameAction.RESET.value]
        if not legal_ids:
            legal_ids = [a.value for a in GameAction if a is not GameAction.RESET]

        chosen_id = self._pick_action(current_hash, legal_ids)
        action = GameAction.from_id(chosen_id)
        self.pending_id = chosen_id

        if action.is_simple():
            self.pending_region = None
            tried, changed = self.action_stats.get(chosen_id, [0, 0])
            n_untried_here = len([a for a in legal_ids if a not in self.tried_from.get(current_hash, set())])
            action.reasoning = (
                f"id={chosen_id} change_rate={changed}/{tried} "
                f"frontier_untried_here={n_untried_here}"
            )
        elif action.is_complex():
            n_regions = 64 // REGION_SIZE
            region_keys = [(rx, ry) for rx in range(n_regions) for ry in range(n_regions)]
            region = self._pick_by_score(self.region_stats, region_keys, self.EPSILON)
            self.pending_region = region
            x = region[0] * REGION_SIZE + random.randint(0, REGION_SIZE - 1)
            y = region[1] * REGION_SIZE + random.randint(0, REGION_SIZE - 1)
            action.set_data({"x": x, "y": y})
            tried, changed = self.region_stats.get(region, [0, 0])
            action.reasoning = {
                "region": str(region),
                "change_rate": f"{changed}/{tried}",
            }

        self.last_hash = current_hash
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"myagent\": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
